# Robustness analyses

This notebook reproduces Appendix B, Appendix E, and Appendix F from the released raw records and supporting inputs.

In [1]:
from pathlib import Path
import csv, hashlib, json, math, re, unicodedata
from collections import Counter, defaultdict
from datetime import date, datetime
from statistics import mean
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display
from scipy.stats import t, ttest_rel

ROOT=Path.cwd()
if not (ROOT/"raw").is_dir(): ROOT=ROOT.parent
RAW_QUERIES=ROOT/"raw"/"query_results"; RAW_AIOS=ROOT/"raw"/"aio_results"
UGC_HOSTS={"facebook.com","instagram.com","linkedin.com","pinterest.com","quora.com","reddit.com","threads.com","threads.net","tiktok.com","twitter.com","x.com","youtube.com","m.youtube.com"}
TRACKING={"gclid","fbclid","dclid","msclkid","mc_cid","mc_eid","igshid"}

def show_table(frame,formats=None):
    output=frame.copy()
    for column,template in (formats or {}).items(): output[column]=output[column].map(template.format)
    display(HTML(output.to_html(index=False,border=0)))

def host(url):
    value=(urlparse(url).hostname or "").lower()
    return value[4:] if value.startswith("www.") else value

def normalize_query(value): return " ".join(unicodedata.normalize("NFKC",str(value or "")).casefold().split())
def normalize_content(value): return " ".join(unicodedata.normalize("NFKC",str(value or "")).casefold().split())

def normalize_key(value):
    value=value.strip().lower().rstrip("/")
    if "://" in value:
        parsed=urlparse(value); h=parsed.hostname or ""; path=parsed.path.strip("/")
    else:
        h,sep,path=value.partition("/"); path=path.strip("/") if sep else ""
    if h.startswith("www."): h=h[4:]
    return h+("/"+path if path else "")

def put_score(mapping,raw_key,score):
    key=normalize_key(raw_key)
    if key in mapping and not math.isclose(mapping[key],score,abs_tol=1e-12):
        if raw_key.strip().lower().split("/",1)[0].startswith("www."): return
    mapping[key]=score

def match_key(url,available):
    parsed=urlparse(url); h=host(url)
    if not h: return None
    parts=[x for x in parsed.path.strip("/").split("/") if x]
    for size in range(len(parts),0,-1):
        key=h+"/"+"/".join(parts[:size])
        if key in available: return key
    if h in available: return h
    labels=h.split(".")
    while len(labels)>2:
        labels=labels[1:]; key=".".join(labels)
        if key in available: return key
    return None

def canonical_url(url):
    parsed=urlparse(url.strip()); h=host(url); path=parsed.path or "/"
    if path!="/": path=path.rstrip("/")
    query=[(k,v) for k,v in parse_qsl(parsed.query,keep_blank_values=True) if k.casefold() not in TRACKING and not k.casefold().startswith("utm_")]
    return urlunparse(("",h+(f":{parsed.port}" if parsed.port else ""),path,"",urlencode(sorted(query)),""))

def paired(values):
    a=np.array([x for x,_ in values]); s=np.array([y for _,y in values]); d=a-s
    margin=t.ppf(.975,len(d)-1)*d.std(ddof=1)/math.sqrt(len(d)); p=ttest_rel(a,s).pvalue
    return len(d),a.mean(),s.mean(),d.mean(),d.mean()-margin,d.mean()+margin,p


## Table 10 — Alternative credibility measures

For each alternative, PC1 and the alternative are matched on the same domain/path keys and URL occurrences. Query means receive equal weight.

In [2]:
pc1={}
for row in csv.DictReader((ROOT/"replication"/"supporting"/"credibility"/"domain_pc1.csv").open()): put_score(pc1,row["domain"],float(row["pc1"]))
mbfc={}; lasser={}
for row in csv.DictReader((ROOT/"replication"/"supporting"/"credibility"/"mbfc_lasser_ratings.csv").open()):
    if row.get("mbfc_fact","").strip(): put_score(mbfc,row["domain"],float(row["mbfc_fact"]))
    if row.get("lewandowsky_mean","").strip(): put_score(lasser,row["domain"],float(row["lewandowsky_mean"]))
burdisso={}
for key,value in json.loads((ROOT/"replication"/"supporting"/"credibility"/"burdisso_reliability_scores.json").read_text()).items(): put_score(burdisso,key,float(value["p-reliability"]))

source_observations=[]
for path in sorted(RAW_AIOS.rglob("result.json")):
    d=json.loads(path.read_text(encoding="utf-8")); serp=[u for u in (d.get("serpLinksDedup") or []) if isinstance(u,str)]
    if not (path.parent/"snapshot.html").is_file(): continue
    if any(host(u)=="google.com" and urlparse(u).path.rstrip("/")=="/goto" for u in serp): continue
    refs=[r["url"] for r in ((d.get("aiOverview") or {}).get("references") or []) if isinstance(r,dict) and r.get("url")]
    source_observations.append((refs,serp))
assert len(source_observations)==7_562

table10=[]
for comparison,alternative,label in [("PC1 & MBFC",mbfc,"MBFC factuality"),("PC1 & Lasser",lasser,"Lasser mean"),("PC1 & Burdisso",burdisso,"Burdisso p-reliability")]:
    shared=set(pc1)&set(alternative); pc1_pairs=[]; alt_pairs=[]
    for refs,serp in source_observations:
        ref_keys=[key for url in refs if (key:=match_key(url,shared))]
        serp_keys=[key for url in serp if (key:=match_key(url,shared))]
        if not ref_keys or not serp_keys: continue
        pc1_pairs.append((mean(pc1[k] for k in ref_keys),mean(pc1[k] for k in serp_keys)))
        alt_pairs.append((mean(alternative[k] for k in ref_keys),mean(alternative[k] for k in serp_keys)))
    for measure,pairs in [("PC1",pc1_pairs),(label,alt_pairs)]:
        n,a,s,g,lo,hi,p=paired(pairs); table10.append([comparison,measure,n,a,s,g,lo,hi,p])

table10=pd.DataFrame(table10,columns=["Comparison","Measure","n","AIO mean","SERP mean","Gap","CI low","CI high","p"])
assert table10.groupby("Comparison")["n"].first().tolist()==[7_174,6_834,7_281]
expected={
    ("PC1 & MBFC","PC1"):(.742,.608,.133),("PC1 & MBFC","MBFC factuality"):(.696,.590,.106),
    ("PC1 & Lasser","PC1"):(.835,.827,.008),("PC1 & Lasser","Lasser mean"):(.886,.874,.013),
    ("PC1 & Burdisso","PC1"):(.791,.786,.005),("PC1 & Burdisso","Burdisso p-reliability"):(.669,.677,-.007),
}
for row in table10.itertuples(index=False): assert tuple(round(x,3) for x in [row._3,row._4,row.Gap])==expected[(row.Comparison,row.Measure)]
show_table(table10,{"n":"{:,}","AIO mean":"{:.3f}","SERP mean":"{:.3f}","Gap":"{:+.3f}","CI low":"{:+.3f}","CI high":"{:+.3f}","p":"{:.3g}"})

Comparison,Measure,n,AIO mean,SERP mean,Gap,CI low,CI high,p
PC1 & MBFC,PC1,"7,281",0.742,0.608,+0.133,+0.130,+0.136,0
PC1 & MBFC,MBFC factuality,"7,281",0.696,0.590,+0.106,+0.103,+0.109,0
PC1 & Lasser,PC1,"6,834",0.835,0.827,+0.008,+0.006,+0.009,1.11e-29
PC1 & Lasser,Lasser mean,"6,834",0.886,0.874,+0.013,+0.011,+0.015,6.68e-31
PC1 & Burdisso,PC1,"7,174",0.791,0.786,+0.005,+0.004,+0.007,1.69e-13
PC1 & Burdisso,Burdisso p-reliability,"7,174",0.669,0.677,-0.007,-0.017,+0.002,0.13


## Repeated-query population

The next cell reads every execution in `raw/query_results/` and joins an AIO only when the same relative record exists in `raw/aio_results/`.

In [3]:
observations=[]
for raw_path in sorted(RAW_QUERIES.rglob("result.json")):
    relative=raw_path.relative_to(RAW_QUERIES); raw=json.loads(raw_path.read_text(encoding="utf-8"))
    date_match=re.search(r"(2026-\d{2}-\d{2})T",relative.parts[0]); day=date_match.group(1)
    query=str(raw.get("keyword") or raw.get("searchKeyword") or "").strip(); qnorm=normalize_query(query)
    aio_path=RAW_AIOS/relative; aio=aio_path.is_file()
    record=dict(relative=relative.as_posix(),date=day,timestamp=str(raw.get("timestamp") or ""),query=query,qnorm=qnorm,aio=int(aio),
                refs=[],source_refs=[],domains=[],ugc=0,pc1=[],ads=0,labels=Counter(),content="",ref_set=set())
    if aio:
        data=json.loads(aio_path.read_text(encoding="utf-8")); ai=data.get("aiOverview") or {}
        refs=[r["url"] for r in (ai.get("references") or []) if isinstance(r,dict) and r.get("url")]; record["refs"]=refs
        record["domains"]=[host(u) for u in refs if host(u)]; record["content"]=normalize_content(ai.get("content") or "")
        record["ref_set"]={canonical_url(u) for u in refs}
        seen=set(); source_refs=[]
        for url in refs:
            key=url.split("#",1)[0]
            if key not in seen: seen.add(key); source_refs.append(url)
        record["source_refs"]=source_refs; record["ugc"]=sum(host(u) in UGC_HOSTS for u in source_refs)
        record["pc1"]=[pc1[key] for u in source_refs if (key:=match_key(u,set(pc1)))]
        for url in refs:
            if host(url) in UGC_HOSTS: continue
            page=aio_path.parent/"page_analysis"/(hashlib.md5(url.encode()).hexdigest()[:8]+".json")
            if page.is_file():
                d=json.loads(page.read_text(encoding="utf-8")); record["ads"]+=int((d.get("ad_metrics") or {}).get("iframe_ad_count") or 0)>0
        verdict=aio_path.parent/"verification_result_grok-4-1-fast-reasoning_v2.json"
        if verdict.is_file(): record["labels"].update(str(v.get("label") or "").upper() for v in json.loads(verdict.read_text(encoding="utf-8")).get("verifications") or [])
    observations.append(record)

assert len(observations)==55_393 and sum(x["aio"] for x in observations)==7_583
by_query=defaultdict(list)
for row in observations: by_query[row["qnorm"]].append(row)
assert len(by_query)==32_608


In [4]:
groups=Counter()
for group in by_query.values():
    day_counts=Counter(row["date"] for row in group); multiple_days=len(day_counts)>1; same_day=any(n>1 for n in day_counts.values())
    key="Once" if len(group)==1 else "Same-day only" if same_day and not multiple_days else "Cross-day only" if multiple_days and not same_day else "Both"
    groups[key]+=1

cross_day={q for q,g in by_query.items() if len({r["date"] for r in g})>1}
cross_exec=sum(len(by_query[q]) for q in cross_day)
day_queries={(r["date"],r["qnorm"]) for r in observations}
same_day_extra=len(observations)-len(day_queries)
assert groups==Counter({"Once":21_605,"Same-day only":3_038,"Cross-day only":6_847,"Both":1_118})
assert (len(cross_day),cross_exec,same_day_extra)==(7_965,27_633,4_918)

repetition=pd.DataFrame([[name,groups[name],100*groups[name]/len(by_query)] for name in ["Once","Same-day only","Cross-day only","Both"]],columns=["Query group","Unique queries","Share"])
show_table(repetition,{"Unique queries":"{:,}","Share":"{:.2f}%"})
display(Markdown(f"Cross-day recurring queries: **{len(cross_day):,} ({len(cross_day)/len(by_query):.2%})**, covering **{cross_exec:,} executions ({cross_exec/len(observations):.2%})**. Same-day repetition adds **{same_day_extra:,} executions ({same_day_extra/len(observations):.2%})**."))

Query group,Unique queries,Share
Once,"21,605",66.26%
Same-day only,"3,038",9.32%
Cross-day only,"6,847",21.00%
Both,"1,118",3.43%


Cross-day recurring queries: **7,965 (24.43%)**, covering **27,633 executions (49.89%)**. Same-day repetition adds **4,918 executions (8.88%)**.

## Table 11 — Query-balanced results

Within each metric’s eligible population, every normalized query receives total weight one.

In [5]:
def ratio(rows,numerator,denominator,balanced=False):
    qcount=Counter(r["qnorm"] for r in rows); num=den=0.0
    for r in rows:
        w=1/qcount[r["qnorm"]] if balanced else 1
        num+=w*numerator(r); den+=w*denominator(r)
    return num/den

aio_rows=[r for r in observations if r["aio"]]; verified=[r for r in aio_rows if sum(r["labels"].values())]

def domain_share(rows,balanced=False):
    qcount=Counter(r["qnorm"] for r in rows); counts=Counter()
    for r in rows:
        w=1/qcount[r["qnorm"]] if balanced else 1
        for domain in r["domains"]: counts[domain]+=w
    return sum(v for _,v in counts.most_common(20))/sum(counts.values())

specs=[
    ("AIO activation (%)",observations,lambda r:r["aio"],lambda r:1,100),
    ("References per AIO",aio_rows,lambda r:len(r["refs"]),lambda r:1,1),
    ("AIO-source UGC share (%)",aio_rows,lambda r:r["ugc"],lambda r:len(r["source_refs"]),100),
    ("AIO-source PC1",aio_rows,lambda r:sum(r["pc1"]),lambda r:len(r["pc1"]),1),
    ("References with ads (%)",aio_rows,lambda r:r["ads"],lambda r:len(r["refs"]),100),
    ("Fidelity (%)",verified,lambda r:r["labels"]["CLEAR"]+r["labels"]["VAGUE"],lambda r:sum(r["labels"].values()),100),
    ("Incorrect claims (%)",verified,lambda r:r["labels"]["INCORRECT"],lambda r:sum(r["labels"].values()),100),
    ("Omitted claims (%)",verified,lambda r:r["labels"]["OMITTED"],lambda r:sum(r["labels"].values()),100),
]
table11=[]
for name,rows,num,den,scale in specs:
    base=scale*ratio(rows,num,den,False); balanced=scale*ratio(rows,num,den,True); table11.append([name,base,balanced,balanced-base])
base=100*domain_share(aio_rows,False); balanced=100*domain_share(aio_rows,True); table11.append(["Top-20 domain share (%)",base,balanced,balanced-base])
table11=pd.DataFrame(table11,columns=["Metric","Baseline","Balanced","Difference"])
expected=[(13.69,16.03),(8.07,7.96),(14.18,14.35),(.7319,.7299),(50.63,52.39),(88.97,89.11),(2.66,2.63),(6.98,6.78),(37.80,37.83)]
for pair,target in zip(table11[["Baseline","Balanced"]].to_numpy(),expected):
    decimals=4 if target[0]<1 else 2
    assert tuple(round(x,decimals) for x in pair)==target
show_table(table11,{"Baseline":"{:.4g}","Balanced":"{:.4g}","Difference":"{:+.4g}"})

Metric,Baseline,Balanced,Difference
AIO activation (%),13.69,16.03,+2.337
References per AIO,8.072,7.964,-0.1081
AIO-source UGC share (%),14.18,14.35,+0.1693
AIO-source PC1,0.7319,0.7299,-0.002054
References with ads (%),50.63,52.39,+1.752
Fidelity (%),88.97,89.11,+0.1444
Incorrect claims (%),2.662,2.632,-0.02991
Omitted claims (%),6.978,6.78,-0.198
Top-20 domain share (%),37.8,37.83,+0.02943


## Variation across repeated observations

In [6]:
pairs=[]
for qnorm,group in by_query.items():
    if len(group)<2: continue
    ordered=sorted(group,key=lambda r:(r["date"],r["timestamp"],r["relative"]))
    for previous,current in zip(ordered,ordered[1:]):
        lag=(date.fromisoformat(current["date"])-date.fromisoformat(previous["date"])).days
        if lag<=0: continue
        row={"qnorm":qnorm,"agreement":previous["aio"]==current["aio"],"both":previous["aio"] and current["aio"]}
        if row["both"]:
            row["same_text"]=previous["content"]==current["content"]
            row["same_sources"]=previous["ref_set"]==current["ref_set"]
            union=previous["ref_set"]|current["ref_set"]
            row["jaccard"]=len(previous["ref_set"]&current["ref_set"])/len(union) if union else 1
        pairs.append(row)

pair_counts=Counter(r["qnorm"] for r in pairs)
for row in pairs: row["weight"]=1/pair_counts[row["qnorm"]]
def weighted(rows,key): return sum(r["weight"]*r[key] for r in rows)/sum(r["weight"] for r in rows)
both=[r for r in pairs if r["both"]]
stability=pd.DataFrame([
    ["AIO-trigger agreement",len(pairs),100*weighted(pairs,"agreement")],
    ["Identical normalized AIO text",len(both),100*weighted(both,"same_text")],
    ["Identical cited-URL set",len(both),100*weighted(both,"same_sources")],
    ["Mean cited-URL-set Jaccard",len(both),weighted(both,"jaccard")],
],columns=["Measure","Pairs","Value"])
assert len(pairs)==17_867
assert round(stability.loc[0,"Value"],2)==95.92 and round(stability.loc[1,"Value"],2)==.06 and round(stability.loc[2,"Value"],2)==.24 and round(stability.loc[3,"Value"],3)==.157
show_table(stability,{"Pairs":"{:,}","Value":"{:.3f}"})

Measure,Pairs,Value
AIO-trigger agreement,"17,867",95.916
Identical normalized AIO text,911,0.061
Identical cited-URL set,911,0.244
Mean cited-URL-set Jaccard,911,0.157


## Appendix F — Real-time-query exclusions

In [7]:
excluded=pd.read_csv(ROOT/"replication"/"supporting"/"realtime_excluded_queries.csv")
appendix_f=excluded.groupby("category",as_index=False).agg(**{"Excluded AIOs":("keyword","size"),"Excluded claims":("n_claims","sum")})
assert {(r.category,r._1,r._2) for r in appendix_f.itertuples(index=False)}=={("Jobs & Education",12,107),("Shopping",1,15),("Technology",12,126)}
show_table(appendix_f)

category,Excluded AIOs,Excluded claims
Jobs & Education,12,107
Shopping,1,15
Technology,12,126
